<a href="https://colab.research.google.com/github/ky537465/Human_Communication_Preferences/blob/main/dataForAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from google.colab import files
import json

uploaded = files.upload()

for fn in uploaded.keys():
  print('Uploaded "{name}"'.format(name=fn, length=len(uploaded[fn])))

Saving formattedData.csv to formattedData.csv
Uploaded "formattedData.csv"


In [22]:
from csv import Error
import sys
q_mushrooms = {
    1: [{"color": "red", "shape": "plain"}, {"color": "green", "shape": "spotted"}, {"color": "blue", "shape": "striped"}],
    2: [{"color": "red", "shape": "striped"}, {"color": "green", "shape": "plain"}, {"color": "red", "shape": "plain"}],
    3: [{"color": "blue", "shape": "spotted"}, {"color": "green", "shape": "spotted"}, {"color": "green", "shape": "striped"}],
    4: [{"color": "red", "shape": "striped"}, {"color": "blue", "shape": "plain"}, {"color": "green", "shape": "spotted"}],
    5: [{"color": "red", "shape": "plain"}, {"color": "green", "shape": "spotted"}, {"color": "blue", "shape": "striped"}],
    6: [{"color": "red", "shape": "striped"}, {"color": "green", "shape": "plain"}, {"color": "red", "shape": "plain"}],
    7: [{"color": "blue", "shape": "spotted"}, {"color": "green", "shape": "spotted"}, {"color": "red", "shape": "striped"}],
    8: [{"color": "red", "shape": "striped"}, {"color": "blue", "shape": "plain"}, {"color": "green", "shape": "spotted"}],
}

q_horizons = {
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 1,
    6: 2,
    7: 3,
    8: 4,
}

data_list = []

raw_data = pd.read_csv('formattedData.csv')

for index, row in raw_data.iterrows():
  #setting = "L"
  #setting = "IMU"
  #setting = "DMU"
  setting = "RND"

  for num in range(1, 9):
    temp = {}
    utt = {}

    if setting == "L":
      temp['action_context'] = q_mushrooms[num]
      temp['workerid'] = row['ID']
      temp['horizon'] = q_horizons[num]
      s = setting + str(num) + ' (T or I)'
      sC1 = setting + str(num) + ' Word1'
      sC2 = setting + str(num) + ' Word2'

      if row[s] == 'Teach':
        utt['type'] = 'description'
        utt['feature'] = row[sC1].lower()
        utt['value'] = int(row[sC2])

      if row[s] == "Instruct":
        utt['color'] = row[sC2].lower()
        utt['shape'] = row[sC1].lower()
        utt['type'] = "instruction"

      temp['utt'] = utt
      utt = {}
    else:
      s = 'NL' + str(num) + ' (T or I)'
      sC1 = 'NL' + str(num) + ' Word1'
      sC2 = 'NL' + str(num) + ' Word2'

      if row['Non-Linear Section (IMU, DMU, RND)'] == setting:
        temp['action_context'] = q_mushrooms[num]
        temp['workerid'] = row['ID']
        temp['horizon'] = q_horizons[num]
        if row[s] == 'Teach':
          utt['type'] = 'description'
          utt['feature'] = row[sC1].lower()
          utt['value'] = int(row[sC2])

        if row[s] == "Instruct":
          try:
            utt['color'] = row[sC2].lower()
          except AttributeError as e:
              utt['color'] = row[sC2]
          utt['shape'] = row[sC1].lower()
          utt['type'] = "instruction"

        temp['utt'] = utt
        utt = {}
    data_list.append(temp)

with open('data_for_agent.json', 'w') as json_file:
    json.dump(data_list, json_file, indent=4)